In [ ]:
# %pip install pandas playwright sofascore-wrapper nest-asyncio
# %playwright install chromium

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search
from IPython.display import display, HTML

async def get_sofa_json(page, url):
    """Helper to bypass 403 blocks and extract raw JSON."""
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)



In [8]:
async def get_target_player(name_query):
    """Searches SofaScore and returns the Player ID and Full Name."""
    api_wrapper = SofascoreAPI()
    search_engine = Search(api_wrapper, search_string=name_query)
    search_results = await search_engine.search_all()
    await api_wrapper.close()
    
    for item in search_results.get('results', []):
        if item.get('type') == 'player':
            return item['entity']['id'], item['entity']['name']
            
    return None, None

In [9]:
async def scrape_player_history(player_id, full_name, start_year=2024):
    start_limit = datetime(start_year, 1, 1)
    end_limit = datetime(2026, 4, 23, 23, 59) # Your target window

    print(f"--- Deep Crawl Initialized for {full_name} ---")
    all_matches_data = []
    processed_match_ids = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
        
        # 1. Fetch the Season Tree
        seasons_url = f"https://api.sofascore.com/api/v1/player/{player_id}/statistics/seasons"
        seasons_data = await get_sofa_json(page, seasons_url)

        for entry in seasons_data.get('uniqueTournamentSeasons', []):
            ut_id = entry['uniqueTournament']['id']
            ut_name = entry['uniqueTournament']['name']
            
            for season in entry.get('seasons', []):
                s_id = season['id']
                s_year = season['year']
                
                # Fast filter for modern seasons
                if not any(yr in s_year for yr in ['24', '25', '26']):
                    continue
                
                print(f"Scanning {ut_name} | Season {s_year}...")
                
                # 2. Fetch matches for this specific tournament/season
                match_list_url = f"https://api.sofascore.com/api/v1/player/{player_id}/unique-tournament/{ut_id}/season/{s_id}/events/last/0"
                
                try:
                    match_batch = await get_sofa_json(page, match_list_url)
                except: continue
                
                for match in match_batch.get('events', []):
                    m_id = match['id']
                    if m_id in processed_match_ids: continue
                    
                    m_date = datetime.fromtimestamp(match.get('startTimestamp', 0))
                    if m_date < start_limit or m_date > end_limit: continue
                    
                    # 3. Fetch Lineups & Active Defenders
                    lineup_url = f"https://api.sofascore.com/api/v1/event/{m_id}/lineups"
                    try:
                        lineup_resp = await get_sofa_json(page, lineup_url)
                        attacker_rating, target_side = None, None
                        
                        for side in ['home', 'away']:
                            for p_node in lineup_resp.get(side, {}).get('players', []):
                                if p_node.get('player', {}).get('id') == player_id:
                                    attacker_rating = p_node.get('statistics', {}).get('rating')
                                    target_side = side
                                    break
                        
                        if attacker_rating:
                            opp_side = 'away' if target_side == 'home' else 'home'
                            opp_team = match[f'{opp_side}Team']['name']
                            
                            active_defs = []
                            for opp in lineup_resp.get(opp_side, {}).get('players', []):
                                d_rating = opp.get('statistics', {}).get('rating')
                                # Must be a defender and actually have a rating
                                if opp.get('position') == 'D' and d_rating:
                                    active_defs.append({'name': opp['player']['name'], 'rating': d_rating})
                            
                            if active_defs:
                                for d in active_defs:
                                    all_matches_data.append({
                                        'Date': m_date.date(),
                                        'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}",
                                        'Attacker_Rating': attacker_rating,
                                        'Defender': d['name'],
                                        'Defender_Team': opp_team,
                                    })
                                processed_match_ids.add(m_id)
                    except: continue

        await browser.close()
    return pd.DataFrame(all_matches_data)

In [11]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search
import nest_asyncio

nest_asyncio.apply()

async def get_sofa_json(page, url):
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)

async def run_targeted_analysis():
    # 1. Dynamic Player Search
    target_query = input("Enter the player to analyze: ")
    
    api_wrapper = SofascoreAPI()
    search_engine = Search(api_wrapper, search_string=target_query)
    search_results = await search_engine.search_all()
    
    player_id, full_name = None, None
    for item in search_results.get('results', []):
        if item.get('type') == 'player':
            player_id = item['entity']['id']
            full_name = item['entity']['name']
            break
    
    await api_wrapper.close()
    
    if not player_id:
        print("Player not found.")
        return None

    # SETTING THE DATE LIMITS
    start_date_limit = datetime(2024, 1, 1)
    end_date_limit = datetime(2026, 4, 23)
    
    print(f"\n--- Researching {full_name} (Range: Jan 2024 - April 2026) ---")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent="Mozilla/5.0")
        
        all_matches_data = []
        last_match_id = 0
        reached_historical_limit = False
        
        while not reached_historical_limit:
            events_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/{last_match_id}"
            print(f"Fetching batch before ID: {last_match_id if last_match_id != 0 else 'Present'}...")
            
            try:
                events_data = await get_sofa_json(page, events_url)
                events = events_data.get('events', [])
            except: break
            
            if not events: break

            for match in events:
                match_ts = match.get('startTimestamp')
                if not match_ts: continue
                
                match_date = datetime.fromtimestamp(match_ts)

                # SKIP if match is newer than our end limit
                if match_date > end_date_limit:
                    continue
                
                # STOP if match is older than our start limit
                if match_date < start_date_limit:
                    print(f"Reached historical limit ({match_date.date()}). Stopping scrape.")
                    reached_historical_limit = True
                    break
                
                match_id = match['id']
                lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
                
                try:
                    lineup_data = await get_sofa_json(page, lineup_url)
                    attacker_rating = None
                    target_side = None
                    
                    # Locate Attacker
                    for side in ['home', 'away']:
                        for p_node in lineup_data.get(side, {}).get('players', []):
                            if p_node.get('player', {}).get('id') == player_id:
                                attacker_rating = p_node.get('statistics', {}).get('rating')
                                target_side = side
                                break
                        if target_side: break
                    
                    if attacker_rating:
                        opp_side = 'away' if target_side == 'home' else 'home'
                        opp_team = match[f'{opp_side}Team']['name']
                        
                        active_defenders = []
                        for opp in lineup_data.get(opp_side, {}).get('players', []):
                            d_rating = opp.get('statistics', {}).get('rating')
                            if opp.get('position') == 'D' and d_rating:
                                active_defenders.append({'name': opp['player']['name'], 'rating': d_rating})
                        
                        if active_defenders:
                            backline_avg = sum(d['rating'] for d in active_defenders) / len(active_defenders)
                            print(f"Logged: {match_date.date()} | vs {opp_team} | Rating: {attacker_rating} | Backline Avg: {backline_avg:.2f}")
                            
                            for defender in active_defenders:
                                all_matches_data.append({
                                    'Date': str(match_date.date()), # Convert to string for JSON compatibility
                                    'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}",
                                    'Attacker_Rating': attacker_rating,
                                    'Defender': defender['name'],
                                    'Defender_Team': opp_team,
                                    'Defender_Rating': defender['rating'],
                                    'Backline_Avg': backline_avg
                                })
                except: continue
            
            # Pagination logic
            last_match_id = events[-1]['id']
            if reached_historical_limit: break
            
        await browser.close()

    if all_matches_data:
        df = pd.DataFrame(all_matches_data)
        
        # Bayesian Math
        global_avg = df['Attacker_Rating'].mean()
        m = 3 # Increased confidence factor for longer date range
        
        stats = df.groupby(['Defender', 'Defender_Team']).agg({
            'Attacker_Rating': 'mean',
            'Defender_Rating': 'mean',
            'Defender': 'count'
        }).rename(columns={'Defender': 'Games'}).reset_index()
        
        stats['Weighted_Difficulty'] = ((stats['Games'] * stats['Attacker_Rating']) + (m * global_avg)) / (stats['Games'] + m)
        
        # --- THE JSON UPGRADE ---
        # Instead of saving a heavy CSV, we compile a lightweight Python dictionary
        # that is instantly readable by any web framework (React, Vue, vanilla JS).
        
        sorted_stats = stats.sort_values(by='Weighted_Difficulty')
        
        web_payload = {
            "player_name": full_name,
            "global_average": round(global_avg, 2),
            "total_matches_analyzed": len(df),
            # Orient="records" turns the dataframe into a list of cleanly formatted rows
            "rankings": sorted_stats.to_dict(orient="records") 
        }

        print(f"\n--- Analysis Complete: {len(df)} 1v1 Data Points ---")
        print("\nTop 5 Hardest Defenders (JSON format ready for webpage):")
        # Pretty-print the top 5 just so you can see the exact structure the UI will receive
        print(json.dumps(web_payload['rankings'][:5], indent=2))
        
        # We return the payload so it stays in memory and can be passed to an API endpoint later
        return web_payload

# Execute
analysis_results = await run_targeted_analysis()


--- Researching Rayan Cherki (Range: Jan 2024 - April 2026) ---
Fetching batch before ID: Present...
Logged: 2025-12-20 | vs West Ham United | Rating: 8 | Backline Avg: 6.23
Logged: 2025-12-27 | vs Nottingham Forest | Rating: 8.4 | Backline Avg: 6.70
Logged: 2026-01-01 | vs Sunderland | Rating: 7.6 | Backline Avg: 7.30
Logged: 2026-01-04 | vs Chelsea | Rating: 6.5 | Backline Avg: 6.56
Logged: 2026-01-07 | vs Brighton & Hove Albion | Rating: 7.9 | Backline Avg: 6.88
Logged: 2026-01-10 | vs Exeter City | Rating: 7.1 | Backline Avg: 4.50
Logged: 2026-01-13 | vs Newcastle United | Rating: 7.7 | Backline Avg: 6.64
Logged: 2026-01-17 | vs Manchester United | Rating: 6.7 | Backline Avg: 7.10
Logged: 2026-01-20 | vs Bodø/Glimt | Rating: 8.1 | Backline Avg: 6.68
Logged: 2026-01-24 | vs Wolverhampton | Rating: 6.4 | Backline Avg: 6.42
Logged: 2026-01-28 | vs Galatasaray | Rating: 8.2 | Backline Avg: 6.48
Logged: 2026-02-01 | vs Tottenham Hotspur | Rating: 8.1 | Backline Avg: 6.25
Logged: 2026-0